In [22]:
# import pandas as pd
# from datetime import timedelta

# # Load files
# interviews = pd.read_excel('nba_postgame_interviews.xlsx')
# gamelogs = pd.read_excel('Player_metrics.xlsx', sheet_name='combined_player_stats')

# # Strip whitespace from column names just in case
# interviews.columns = interviews.columns.str.strip()
# gamelogs.columns = gamelogs.columns.str.strip()

# # Standardize date columns to datetime
# interviews['Upload Date'] = pd.to_datetime(interviews['Upload Date'])
# gamelogs['Date'] = pd.to_datetime(gamelogs['Date'])

# # Drop rows where player did not play
# dnp_values = ['Inactive', 'Did Not Play', 'Did Not Dress']
# gamelogs = gamelogs[~gamelogs['GmSc'].isin(dnp_values)]
# gamelogs['GmSc'] = pd.to_numeric(gamelogs['GmSc'], errors='coerce')
# gamelogs = gamelogs.dropna(subset=['GmSc'])

# # Sort gamelogs by player and date
# gamelogs = gamelogs.sort_values(['Player', 'Date']).reset_index(drop=True)

# # Filter to only postgame interviews (uploaded exactly 1 day after a game)
# def is_postgame_interview(player, upload_date):
#     player_games = gamelogs[gamelogs['Player'] == player]
#     game_dates = player_games['Date'].values
#     expected_game_date = upload_date - pd.Timedelta(days=1)
#     return expected_game_date in game_dates

# interviews['is_postgame'] = interviews.apply(
#     lambda row: is_postgame_interview(row['Name'], row['Upload Date']),
#     axis=1
# )

# # Show excluded rows for manual verification before filtering
# excluded = interviews[~interviews['is_postgame']][['Name', 'Upload Date', 'Link']]
# print(f"Total interviews: {len(interviews)}")
# print(f"Postgame interviews: {interviews['is_postgame'].sum()}")
# print(f"Excluded: {(~interviews['is_postgame']).sum()}")
# print("\nExcluded rows:")
# print(excluded)

# # Filter out non-postgame interviews
# interviews = interviews[interviews['is_postgame']].drop(columns='is_postgame')

# # Function to compute both metrics for a given player + interview date
# def get_gmsc_metrics(player, upload_date):
#     player_games = gamelogs[gamelogs['Player'] == player]
    
#     # The game from the interview night is 1 day before upload date
#     interview_game_date = upload_date - pd.Timedelta(days=1)
    
#     # Historical average: all games up to and including the interview game
#     historical = player_games[player_games['Date'] <= interview_game_date]
#     if historical.empty:
#         historical_avg = None
#     else:
#         historical_avg = round(historical['GmSc'].mean(), 2)
    
#     # Next game GmSc: first game strictly after the interview game
#     next_games = player_games[player_games['Date'] > interview_game_date]
#     if next_games.empty:
#         next_gmsc = None
#     else:
#         next_gmsc = next_games.iloc[0]['GmSc']
    
#     return historical_avg, next_gmsc

# # Apply to every row
# interviews[['historical_gmsc_avg', 'next_game_gmsc']] = interviews.apply(
#     lambda row: pd.Series(get_gmsc_metrics(row['Name'], row['Upload Date'])),
#     axis=1
# )

# # Save to new file
# interviews.to_excel('nba_postgame_interviews_with_gmsc.xlsx', index=False)

# print("\nDone! Output saved to nba_postgame_interviews_with_gmsc.xlsx")
# print(interviews[['Name', 'Upload Date', 'historical_gmsc_avg', 'next_game_gmsc']].head(10))

Total interviews: 1166
Postgame interviews: 610
Excluded: 556

Excluded rows:
                Name Upload Date                                         Link
1     Andrew Wiggins  2022-06-07  https://www.youtube.com/watch?v=oBeViBdgB-c
2     Andrew Wiggins  2022-06-04  https://www.youtube.com/watch?v=TS_eC7qy95I
4     Andrew Wiggins  2022-06-02  https://www.youtube.com/watch?v=5I_aqReINGc
5     Andrew Wiggins  2022-05-30  https://www.youtube.com/watch?v=EkCwn7gysaY
8     Andrew Wiggins  2022-05-22  https://www.youtube.com/watch?v=dY1jj2iN3_M
...              ...         ...                                          ...
1161    Terance Mann  2021-02-13  https://www.youtube.com/watch?v=9E_kVjioieM
1162    Terance Mann  2021-01-29  https://www.youtube.com/watch?v=OmS0zz_UinU
1163    Terance Mann  2021-01-19  https://www.youtube.com/watch?v=78C8oQUs0SU
1164    Terance Mann  2020-12-28  https://www.youtube.com/watch?v=EmzW3OtBWMA
1165    Terance Mann  2020-12-07  https://www.youtube.com/watch?

In [7]:
import pandas as pd
from datetime import timedelta

# Load files
interviews = pd.read_excel('nba_postgame_interviews.xlsx')
gamelogs = pd.read_excel('Player_metrics.xlsx', sheet_name='combined_player_stats')

# Strip whitespace from column names
interviews.columns = interviews.columns.str.strip()
gamelogs.columns = gamelogs.columns.str.strip()

# Standardize date columns to datetime
interviews['Upload Date'] = pd.to_datetime(interviews['Upload Date'])
gamelogs['Date'] = pd.to_datetime(gamelogs['Date'])

# Drop rows where player did not play
dnp_values = ['Inactive', 'Did Not Play', 'Did Not Dress']
gamelogs = gamelogs[~gamelogs['GmSc'].isin(dnp_values)]
gamelogs['GmSc'] = pd.to_numeric(gamelogs['GmSc'], errors='coerce')
gamelogs = gamelogs.dropna(subset=['GmSc'])

# Sort gamelogs by player and date
gamelogs = gamelogs.sort_values(['Player', 'Date']).reset_index(drop=True)

# Filter to only postgame interviews (uploaded exactly 1 day after a game)
def is_postgame_interview(player, upload_date):
    player_games = gamelogs[gamelogs['Player'] == player]
    game_dates = player_games['Date'].values
    expected_game_date = upload_date - pd.Timedelta(days=1)
    return expected_game_date in game_dates

interviews['is_postgame'] = interviews.apply(
    lambda row: is_postgame_interview(row['Name'], row['Upload Date']),
    axis=1
)

# Show excluded rows
excluded = interviews[~interviews['is_postgame']][['Name', 'Upload Date', 'Link']]
print(f"Total interviews: {len(interviews)}")
print(f"Postgame interviews: {interviews['is_postgame'].sum()}")
print(f"Excluded: {(~interviews['is_postgame']).sum()}")

# Filter out non-postgame interviews
interviews = interviews[interviews['is_postgame']].drop(columns='is_postgame')

# Function to compute gmsc metrics + next game opponent stats
def get_metrics(player, upload_date):
    player_games = gamelogs[gamelogs['Player'] == player]
    
    # Find the actual game date — the most recent game on or before the upload date
    prior_games = player_games[player_games['Date'] <= upload_date]
    if prior_games.empty:
        return None, None, None, None, None, None
    
    interview_game_date = prior_games.iloc[-1]['Date']

    # Historical average: all games up to and including the interview game
    historical = player_games[player_games['Date'] <= interview_game_date]
    historical_avg = round(historical['GmSc'].mean(), 2) if not historical.empty else None

    # Next game: first game strictly after the interview game
    next_games = player_games[player_games['Date'] > interview_game_date]
    if next_games.empty:
        return historical_avg, None, None, None, None, None

    next_game = next_games.iloc[0]
    next_gmsc = next_game['GmSc']
    next_date = next_game['Date']

    # Days of rest = days between actual game date and next game date minus 1
    days_of_rest = (next_date - interview_game_date).days - 1

    # Opponent stats from next game row
    next_opp = next_game['Opp'] if 'Opp' in next_game.index else None
    drtg = next_game['DRtg'] if 'DRtg' in next_game.index else None
    win_pct = next_game['WinPct'] if 'WinPct' in next_game.index else None

    return historical_avg, next_gmsc, days_of_rest, next_opp, drtg, win_pct

# Apply to every row
interviews[['historical_gmsc_avg', 'next_game_gmsc', 'days_of_rest', 'opp', 'opp_drtg', 'opp_win_pct']] = interviews.apply(
    lambda row: pd.Series(get_metrics(row['Name'], row['Upload Date'])),
    axis=1
)

# Save
interviews.to_excel('nba_postgame_interviews_with_gmsc.xlsx', index=False)

print("\nDone! Output saved to nba_postgame_interviews_with_gmsc.xlsx")
print(interviews[['Name', 'Upload Date', 'historical_gmsc_avg', 'next_game_gmsc', 'days_of_rest', 'opp', 'opp_drtg', 'opp_win_pct']].head(10))

Total interviews: 1166
Postgame interviews: 610
Excluded: 556

Done! Output saved to nba_postgame_interviews_with_gmsc.xlsx
              Name Upload Date  historical_gmsc_avg  next_game_gmsc  \
0   Andrew Wiggins  2022-06-09                12.08            13.1   
3   Andrew Wiggins  2022-06-03                12.16             5.4   
6   Andrew Wiggins  2022-05-27                12.12            15.5   
7   Andrew Wiggins  2022-05-23                12.12             9.5   
9   Andrew Wiggins  2022-05-19                11.99            11.9   
11  Andrew Wiggins  2022-05-10                12.09             0.4   
12  Andrew Wiggins  2022-05-04                12.00            15.5   
17  Andrew Wiggins  2022-03-24                12.25             6.1   
19  Andrew Wiggins  2022-03-13                12.31            13.7   
20  Andrew Wiggins  2022-03-02                12.44            11.8   

    days_of_rest  opp  opp_drtg  opp_win_pct  
0            1.0  BOS     106.9     0.621951  
